<a href="https://colab.research.google.com/github/RakyKXD/WithList/blob/main/InvokeAI-Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **InvokeAI - Barren Wardo**
> Note: Probably will only work on Paid Colab.

### How to start?
1.   Run Setup Cell
2.   Run Launcher

Enjoy! ❤️

---

## **Setup**
> Note :
> 1. When prompted, click "Restart".
> 2. Remove # in the 2nd last line to use beta version of InvokeAI.

In [ ]:
# Create directory and install required system dependencies
!mkdir -p /content/invokeai
!sudo apt update -y && sudo apt install -y python3.10 python3.10-venv python3.10-dev libglib2.0-0 libgl1-mesa-glx build-essential python3-opencv libopencv-dev wget curl

# Download and install cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

# Create Python 3.10 virtual environment
!python3.10 -m venv /content/invokeai_venv

# Install InvokeAI directly into the virtual environment
!/content/invokeai_venv/bin/pip install --upgrade pip wheel setuptools
!/content/invokeai_venv/bin/pip install "InvokeAI[xformers]" --use-pep517 --extra-index-url https://download.pytorch.org/whl/cu121

## **Launch InvokeAI**

### **LocalTunnel**
> Note : Copy the password IP then go to the mentioned url & paste it there.

In [ ]:
import subprocess
import threading
import time
import socket
import urllib.request

def iframe_thread(port):
    while True:
        time.sleep(0.5)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        result = sock.connect_ex(('127.0.0.1', port))
        if result == 0:
            sock.close()
            break
        sock.close()

    print("\nInvokeAI finished loading, trying to launch localtunnel (if it gets stuck here, localtunnel may have issues)\n")
    print("The password/endpoint IP for localtunnel is:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip("\n"))

    p = subprocess.Popen(["lt", "--port", str(port)], stdout=subprocess.PIPE)
    for line in p.stdout:
        print(line.decode(), end='')

threading.Thread(target=iframe_thread, args=(9090,), daemon=True).start()

!invokeai-web --root /content/invokeai

### **Cloudflare**

In [ ]:
import subprocess
import threading
import time
import socket

def tunnel_thread(port=9090):
    while True:
        time.sleep(1)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        result = sock.connect_ex(('127.0.0.1', port))
        sock.close()
        if result == 0:
            break

    print("\n[✓] InvokeAI server detected on port 9090. Starting Cloudflare tunnel...\n")
    p = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}"],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )
    for line in p.stderr:
        if "trycloudflare.com" in line:
            url_start = line.find("https://")
            if url_start != -1:
                print(f"\n==========================================\nInvokeAI Public URL: {line[url_start:].strip()}\n==========================================\n")
                break

# Start tunnel monitor in background
threading.Thread(target=tunnel_thread, daemon=True, args=(9090,)).start()

# Launch InvokeAI directly using the venv binary
!/content/invokeai_venv/bin/invokeai-web --root /content/invokeai --host 0.0.0.0 --port 9090